In [103]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle

In [104]:
dataset= pd.read_csv('Churn_Modelling.csv')
dataset.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [105]:
## preprocess the not required columns
dataset.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1, inplace=True)


In [106]:
dataset.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [107]:
## applying encoding for categorical columns
le = LabelEncoder()
dataset['Gender']= le.fit_transform(dataset['Gender'])



In [108]:
dataset.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


In [109]:
## One hot encoding for categorical columns
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder()  
geo_encoded = ohe.fit_transform(dataset[['Geography']])
geo_encoded

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [110]:
ohe.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [111]:
geo_encoded_df = pd.DataFrame(geo_encoded.toarray(), columns=ohe.get_feature_names_out(['Geography']))

In [112]:
geo_encoded_df.head()

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0


In [113]:
## combine one hot encoded columns with the original dataset
dataset= pd.concat([dataset, geo_encoded_df], axis=1).drop(['Geography'], axis=1)
dataset.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [ ]:
with open('label_encoder.pkl', 'wb') as file:
    pickle.dump(le, file)



with open('one_hot_encoder.pkl', 'wb') as file:
    pickle.dump(ohe, file)


In [115]:
dataset.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [116]:
x= dataset.drop('Exited', axis=1)
y=dataset['Exited']

In [117]:
x.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0.0,0.0,1.0


In [ ]:
## divide the dataset into train and validation sets
X = dataset.drop('Exited', axis=1)
y = dataset['Exited']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

## scale the train and validation data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# keep target values in the shape expected by Keras
y_train = y_train.astype('float32').to_numpy().reshape(-1, 1)
y_val = y_val.astype('float32').to_numpy().reshape(-1, 1)

In [121]:
y_train.shape, y_val.shape, y_test.shape

((8000, 1), (1000, 1), (1000, 1))

In [122]:
y_train

array([[0.],
       [0.],
       [1.],
       ...,
       [1.],
       [1.],
       [0.]], dtype=float32)

In [ ]:
with open('standard_scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

## ANN implementation
### Dense: Sequential network, for hidden unit we use Dense(# units), Activation function (sigmoid, tanh, Relu, leakyRelu), Optimiser for backpropogration, Loss function, metrics (accurancy, mse, mae), training log visualization

In [90]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [91]:
(X_train_scaled.shape[1], )

(12,)

In [92]:
## Build a slightly smaller and more stable model
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

In [93]:
model.summary()

Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_12 (Dense)            (None, 64)                832       
                                                                 
 dense_13 (Dense)            (None, 32)                2080      
                                                                 
 dense_14 (Dense)            (None, 16)                528       
                                                                 
 dense_15 (Dense)            (None, 1)                 17        
                                                                 
Total params: 3457 (13.50 KB)
Trainable params: 3457 (13.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [94]:
## compile model with a more stable optimizer and learning rate
model.compile(optimizer=Adam(learning_rate=0.0001), loss='binary_crossentropy', metrics=['accuracy'])

In [95]:
## setup the tensorboard callback
log_dir= "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
log_dir

'logs/fit/20260707-214202'

In [96]:
tensorflow_callback= TensorBoard(log_dir=log_dir, histogram_freq=1)

In [97]:
## setup early stopping callback
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)

In [98]:
history = model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=30,
    batch_size=32,
    callbacks=[tensorflow_callback, early_stopping_callback],
    verbose=1
)

Epoch 1/30
 15/250 [>.............................] - ETA: 0s - loss: 0.7025 - accuracy: 0.6042  

2026-07-07 21:42:02.605560: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node Adam/AssignAddVariableOp.


250/250 [==============================] - 1s 4ms/step - loss: 0.6337 - accuracy: 0.6731 - val_loss: 0.5741 - val_accuracy: 0.7140
Epoch 2/30
250/250 [==============================] - 1s 3ms/step - loss: 0.5428 - accuracy: 0.7604 - val_loss: 0.5036 - val_accuracy: 0.7910
Epoch 3/30
250/250 [==============================] - 1s 3ms/step - loss: 0.4942 - accuracy: 0.7876 - val_loss: 0.4652 - val_accuracy: 0.8030
Epoch 4/30
250/250 [==============================] - 1s 3ms/step - loss: 0.4688 - accuracy: 0.7933 - val_loss: 0.4441 - val_accuracy: 0.8110
Epoch 5/30
250/250 [==============================] - 1s 3ms/step - loss: 0.4539 - accuracy: 0.7956 - val_loss: 0.4317 - val_accuracy: 0.8130
Epoch 6/30
250/250 [==============================] - 1s 5ms/step - loss: 0.4449 - accuracy: 0.8001 - val_loss: 0.4243 - val_accuracy: 0.8140
Epoch 7/30
250/250 [==============================] - 1s 5ms/step - loss: 0.4399 - accuracy: 0.8040 - val_loss: 0.4186 - val_accuracy: 0.8180
Epoch 8/30
250/25

In [99]:
model.save('churn_model.h5')


/Users/shikhar/Documents/Churn_modeling_with_ann/venv/lib/python3.11/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [100]:
## load tensorboard extension
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [102]:
tensorboard --logdir logs/fit

Reusing TensorBoard on port 6006 (pid 48593), started 8:12:13 ago. (Use '!kill 48593' to kill it.)

In [ ]:
## load the model and make predictions

